# Validate Strategy

Stress-test a strategy before committing it to paper trading.
Answers three questions:

1. **Plateau detection** — Are the best params on a robust plateau, or a fragile spike?
2. **Walk-forward analysis** — Do optimized params work on data the optimizer hasn't seen?
3. **Bootstrap confidence intervals** — How much of the result depends on a few lucky trades?

**Prerequisites:** Run a parameter sweep first (e.g. `backtest_ema_cross.ipynb`).

**How to use for a different strategy:** Change Cell 1 — the sweep file path,
the strategy factory, and the param combos. Everything else is strategy-agnostic.

In [ ]:
# ── Cell 1: Imports + config ───────────────────────────────────────
#
# ★ This is the only cell you change per strategy. ★

from decimal import Decimal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from nautilus_trader.model.data import BarType
from nautilus_trader.model.identifiers import InstrumentId, Venue
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.backtesting import make_engine, run_walk_forward, rolling_performance, tag_regimes, performance_by_regime, run_fee_sweep
from src.core import bar_type_str, with_venue_config
from src.strategies.bb_meanrev import BBMeanRev, BBMeanRevConfig
from src.strategies.donchian_breakout import DonchianBreakout, DonchianBreakoutConfig
from src.strategies.ema_cross_atr import EMACrossATR, EMACrossATRConfig
from src.strategies.ema_cross_bracket import EMACrossBracket, EMACrossBracketConfig
from src.strategies.ema_cross_stop_entry import EMACrossStopEntry, EMACrossStopEntryConfig
from src.strategies.ema_cross_tp import EMACrossTP, EMACrossTPConfig
from src.strategies.ma_cross import MACross, MACrossConfig
from src.strategies.ma_cross_long_only import MACrossLongOnly, MACrossLongOnlyConfig
from src.strategies.ma_cross_trailing_stop import MACrossTrailingStop, MACrossTrailingStopConfig
from src.strategies.macd_rsi import MACDRSI, MACDRSIConfig

from charts import plot_rolling_pnl, plot_fee_sensitivity, plot_regime_overlay
from utils import make_instrument_id, save_notebook, save_notebook_html

# ── What to validate ──────────────────────────────────────────────
EXCHANGE         = "BINANCE"
ASSET            = "BTC"
BAR_INTERVAL     = "1d"
STARTING_CAPITAL = 10_000
TRADE_SIZE       = 1000              # $1,000 USD per trade
LEVERAGE         = 20                # Backtesting leverage (margin_init = 1/20 = 5%)

STRATEGY    = 'EMACrossBracket'
N_RESAMPLES = 10_000

# Standard
TRADE_NOTIONAL = Decimal(TRADE_SIZE)
INSTRUMENT_ID  = make_instrument_id(ASSET, EXCHANGE)
TRADING_PAIR   = INSTRUMENT_ID.split("-")[0]
BAR_TYPE_STR   = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)
VENUE          = Venue(EXCHANGE)

# Files
CATALOG_PATH = "../data/catalog"
SWEEP_FILE   = f"../data/sweeps/{STRATEGY}_{INSTRUMENT_ID}_{BAR_INTERVAL}.parquet"
RESULT_NAME  = f"validate_{STRATEGY}_{INSTRUMENT_ID}_{BAR_INTERVAL}"

# ── Strategy registry ─────────────────────────────────────────────
# (StrategyClass, ConfigClass, param_map, fixed_params)
# param_map: only keys that differ from config field names.
# Keys that already match (bb_period, atr_period, etc.) pass through.
# fixed_params (optional): extra kwargs always passed to config.
STRATEGIES = {
    "MACross-EMA":          (MACross, MACrossConfig, {"fast": "fast_period", "slow": "slow_period"}, {"ma_type": "EMA"}),
    "MACross-SMA":          (MACross, MACrossConfig, {"fast": "fast_period", "slow": "slow_period"}, {"ma_type": "SMA"}),
    "MACross-HMA":          (MACross, MACrossConfig, {"fast": "fast_period", "slow": "slow_period"}, {"ma_type": "HMA"}),
    "MACross-DEMA":         (MACross, MACrossConfig, {"fast": "fast_period", "slow": "slow_period"}, {"ma_type": "DEMA"}),
    "MACross-AMA":          (MACross, MACrossConfig, {"fast": "fast_period", "slow": "slow_period"}, {"ma_type": "AMA"}),
    "MACross-VIDYA":        (MACross, MACrossConfig, {"fast": "fast_period", "slow": "slow_period"}, {"ma_type": "VIDYA"}),
    "MACrossLongOnly":      (MACrossLongOnly, MACrossLongOnlyConfig, {"fast": "fast_period", "slow": "slow_period"}, {"ma_type": "EMA"}),
    "MACrossTrailingStop":  (MACrossTrailingStop, MACrossTrailingStopConfig, {"fast": "fast_period", "slow": "slow_period", "trailing_mult": "trailing_atr_multiple"}, {"ma_type": "EMA"}),
    "EMACrossATR":          (EMACrossATR, EMACrossATRConfig, {"fast": "fast_ema_period", "slow": "slow_ema_period", "atr_sl": "atr_sl_multiplier", "atr_tp": "atr_tp_multiplier"}),
    "EMACrossBracket":      (EMACrossBracket, EMACrossBracketConfig, {"fast": "fast_ema_period", "slow": "slow_ema_period", "bracket_dist": "bracket_distance_atr"}),
    "EMACrossStopEntry":    (EMACrossStopEntry, EMACrossStopEntryConfig, {"fast": "fast_ema_period", "slow": "slow_ema_period", "trail_mult": "trailing_atr_multiple"}),
    "EMACrossTP":           (EMACrossTP, EMACrossTPConfig, {"fast": "fast_ema_period", "slow": "slow_ema_period"}),
    "BBMeanRev":            (BBMeanRev, BBMeanRevConfig, {"rsi_buy": "rsi_buy_threshold", "rsi_sell": "rsi_sell_threshold"}),
    "MACDRSI":              (MACDRSI, MACDRSIConfig, {"fast": "macd_fast_period", "slow": "macd_slow_period", "signal": "macd_signal_period", "rsi_ob": "rsi_overbought", "rsi_os": "rsi_oversold", "rsi_entry": "rsi_entry_threshold"}),
    "DonchianBreakout":     (DonchianBreakout, DonchianBreakoutConfig, {}),
}

def strategy_factory(eng, params):
    entry = STRATEGIES[STRATEGY]
    cls, cfg_cls, param_map = entry[0], entry[1], entry[2]
    fixed = entry[3] if len(entry) > 3 else {}
    mapped = {param_map.get(k, k): v for k, v in params.items()}
    cfg = cfg_cls(
        instrument_id=InstrumentId.from_str(INSTRUMENT_ID),
        bar_type=BarType.from_str(BAR_TYPE_STR),
        trade_notional=TRADE_NOTIONAL,
        **fixed,
        **mapped,
    )
    eng.add_strategy(cls(cfg))

# ── Param grid (strategy-specific) ──────────────────────────────
match STRATEGY:
    case 'MACross-EMA' | 'MACross-SMA' | 'MACross-HMA' | 'MACross-DEMA' | 'MACross-AMA' | 'MACross-VIDYA' | 'MACrossLongOnly' | 'EMACrossATR' | 'EMACrossTP':
        FAST_PERIODS = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100]
        SLOW_PERIODS = [10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100, 200]
        PARAM_COMBOS = [{"fast": f, "slow": s} for f in FAST_PERIODS for s in SLOW_PERIODS if f < s]
        ROW_PARAM = "slow"
        COL_PARAM = "fast"
    case 'MACrossTrailingStop':
        FAST_PERIODS = [5, 10, 15, 20, 25]
        SLOW_PERIODS = [15, 20, 30, 40, 50]
        PARAM_COMBOS = [{"fast": f, "slow": s} for f in FAST_PERIODS for s in SLOW_PERIODS if f < s]
        ROW_PARAM = "slow"
        COL_PARAM = "fast"
    case 'EMACrossBracket' | 'EMACrossStopEntry':
        FAST_PERIODS = [5, 8, 10, 12, 15, 20, 25, 30, 35, 40, 45, 50]
        SLOW_PERIODS = [10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100, 200]
        PARAM_COMBOS = [{"fast": f, "slow": s} for f in FAST_PERIODS for s in SLOW_PERIODS if f < s]
        ROW_PARAM = "slow"
        COL_PARAM = "fast"
    case 'BBMeanRev':
        BB_PERIODS = [15, 20, 25, 30]
        BB_STDS = [1.5, 2.0, 2.5, 3.0]
        PARAM_COMBOS = [{"bb_period": p, "bb_std": s} for p in BB_PERIODS for s in BB_STDS]
        ROW_PARAM = "bb_std"
        COL_PARAM = "bb_period"
    case 'MACDRSI':
        FAST_PERIODS = [8, 12, 16]
        SLOW_PERIODS = [20, 26, 34]
        PARAM_COMBOS = [{"fast": f, "slow": s} for f in FAST_PERIODS for s in SLOW_PERIODS if f < s]
        ROW_PARAM = "slow"
        COL_PARAM = "fast"
    case 'DonchianBreakout':
        PERIODS = [10, 15, 20, 25, 30, 40, 50]
        PARAM_COMBOS = [{"dc_period": p} for p in PERIODS]
        ROW_PARAM = "dc_period"
        COL_PARAM = "dc_period"
    case _:
        raise ValueError(f"No param grid for {STRATEGY!r}")

print(f"Strategy : {STRATEGY}")
print(f"Asset    : {INSTRUMENT_ID}")
print(f"Interval : {BAR_INTERVAL}")
print(f"Combos   : {len(PARAM_COMBOS)}")
print("Imports OK")

In [ ]:
# ── Cell 2: Load data ──────────────────────────────────────────────
catalog    = ParquetDataCatalog(CATALOG_PATH)
instrument = catalog.instruments(instrument_ids=[INSTRUMENT_ID])[0]
bars       = catalog.bars(bar_types=[BAR_TYPE_STR])

# Override margins for backtesting (catalog instruments have raw/live margins)
instrument = with_venue_config(instrument, LEVERAGE)

# Settlement currency for PnL queries (USDC for HL, USDT for Binance)
CURRENCY = instrument.settlement_currency

print(f"Instrument : {instrument.id}")
print(f"Currency   : {CURRENCY}")
print(f"Leverage   : {LEVERAGE}x (margin_init={instrument.margin_init}, margin_maint={instrument.margin_maint})")
print(f"Bars       : {len(bars):,}")
print(f"Range      : {pd.Timestamp(bars[0].ts_event, unit='ns', tz='UTC'):%Y-%m-%d}")
print(f"           → {pd.Timestamp(bars[-1].ts_event, unit='ns', tz='UTC'):%Y-%m-%d}")

---
## Part 1: Parameter Plateau Detection

A sharp PnL spike at one param combo surrounded by losers = fragile, likely overfit.
A broad profitable region where neighbours also win = robust, more likely to hold out-of-sample.

We load the saved sweep and measure how "smooth" the profitable region is.

In [ ]:
# ── Cell 3: Load sweep + find best params ──────────────────────────
sweep_df = pd.read_parquet(SWEEP_FILE)
print(f"Loaded {len(sweep_df)} rows from {SWEEP_FILE}")

best_idx = sweep_df["total_pnl"].idxmax()
best_row = sweep_df.loc[best_idx]
param_keys = list(PARAM_COMBOS[0].keys())
best_params = {k: best_row[k].item() if hasattr(best_row[k], "item") else best_row[k] for k in param_keys}
best_pnl = best_row["total_pnl"]
param_str = ", ".join(f"{k}={v}" for k, v in best_params.items())
print(f"Best combo: {param_str}  PnL={best_pnl:,.2f}")

In [ ]:
# ── Cell 4: Plateau analysis ──────────────────────────────────────
#
# For each param combo, count what fraction of its immediate
# neighbours in the grid are also profitable.  High = plateau.

def plateau_scores(df, row_col, col_col, value_col="total_pnl"):
    """Compute neighbour-profitability score for each cell in a 2D grid.

    Returns a copy of df with added columns:
      - profitable:       bool, is this combo profitable?
      - neighbour_score:  fraction of neighbours (including self) that are profitable (0-1)
      - neighbour_avg:    mean PnL of neighbours (including self)
    """
    pivot = df.pivot(index=row_col, columns=col_col, values=value_col)
    row_vals = list(pivot.index)
    col_vals = list(pivot.columns)

    scores = {}
    for ri, rv in enumerate(row_vals):
        for ci, cv in enumerate(col_vals):
            val = pivot.iloc[ri, ci]
            if pd.isna(val):
                continue
            # Gather neighbours (including self) — 3x3 window
            nbrs = []
            for dr in [-1, 0, 1]:
                for dc in [-1, 0, 1]:
                    nr, nc = ri + dr, ci + dc
                    if 0 <= nr < len(row_vals) and 0 <= nc < len(col_vals):
                        nv = pivot.iloc[nr, nc]
                        if not pd.isna(nv):
                            nbrs.append(nv)
            n_profitable = sum(1 for n in nbrs if n > 0)
            scores[(rv, cv)] = {
                "neighbour_score": n_profitable / len(nbrs) if nbrs else 0,
                "neighbour_avg": np.mean(nbrs) if nbrs else 0,
                "neighbour_count": len(nbrs),
            }

    result = df.copy()
    result["profitable"] = result[value_col] > 0
    result["neighbour_score"] = result.apply(
        lambda r: scores.get((r[row_col], r[col_col]), {}).get("neighbour_score", 0), axis=1
    )
    result["neighbour_avg"] = result.apply(
        lambda r: scores.get((r[row_col], r[col_col]), {}).get("neighbour_avg", 0), axis=1
    )
    return result


scored = plateau_scores(sweep_df, ROW_PARAM, COL_PARAM)

# Best combo's plateau score
best_score = scored.loc[best_idx, "neighbour_score"]
best_avg   = scored.loc[best_idx, "neighbour_avg"]

# Overall landscape stats
n_profitable    = scored["profitable"].sum()
n_total         = len(scored)
pct_profitable  = n_profitable / n_total * 100
median_score    = scored.loc[scored["profitable"], "neighbour_score"].median()

print(f"Grid landscape:")
print(f"  {n_profitable}/{n_total} combos profitable ({pct_profitable:.0f}%)")
print(f"  Median neighbour score (profitable combos): {median_score:.2f}")
print()
print(f"Best combo ({COL_PARAM}={best_row[COL_PARAM]}, {ROW_PARAM}={best_row[ROW_PARAM]}):")
print(f"  PnL:             {best_pnl:,.2f}")
print(f"  Neighbour score: {best_score:.2f}  (1.0 = all neighbours profitable)")
print(f"  Neighbour avg:   {best_avg:,.2f}")
print()

if best_score >= 0.8:
    print("✅ PLATEAU — best params sit in a robust profitable region.")
elif best_score >= 0.5:
    print("⚠️  RIDGE — best params are on the edge of profitability. Moderate risk.")
else:
    print("🚩 SPIKE — best params are an isolated peak. High overfit risk.")

In [ ]:
# ── Cell 5: Plateau heatmap ───────────────────────────────────────
#
# Shows neighbour score as a heatmap — bright green = robust plateau.

pivot_score = scored.pivot(
    index=ROW_PARAM, columns=COL_PARAM, values="neighbour_score"
)
pivot_pnl = scored.pivot(
    index=ROW_PARAM, columns=COL_PARAM, values="total_pnl"
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: PnL heatmap (existing view)
ax = axes[0]
vmax = max(abs(np.nanmin(pivot_pnl.values)), abs(np.nanmax(pivot_pnl.values)))
im = ax.imshow(pivot_pnl.values, cmap="RdYlGn", aspect="auto", vmin=-vmax, vmax=vmax)
ax.set_xticks(range(len(pivot_pnl.columns)))
ax.set_xticklabels([str(c) for c in pivot_pnl.columns], fontsize=8)
ax.set_yticks(range(len(pivot_pnl.index)))
ax.set_yticklabels([str(i) for i in pivot_pnl.index], fontsize=8)
ax.set_xlabel(COL_PARAM.title())
ax.set_ylabel(ROW_PARAM.title())
ax.set_title("PnL")
fig.colorbar(im, ax=ax, shrink=0.8)

# Right: Neighbour score heatmap (new view)
ax = axes[1]
im2 = ax.imshow(pivot_score.values, cmap="YlGn", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(pivot_score.columns)))
ax.set_xticklabels([str(c) for c in pivot_score.columns], fontsize=8)
ax.set_yticks(range(len(pivot_score.index)))
ax.set_yticklabels([str(i) for i in pivot_score.index], fontsize=8)
ax.set_xlabel(COL_PARAM.title())
ax.set_ylabel(ROW_PARAM.title())
ax.set_title("Neighbour Score (1.0 = all neighbours profitable)")
fig.colorbar(im2, ax=ax, shrink=0.8)

# Mark the best combo on both
best_col_val = best_row[COL_PARAM]
best_row_val = best_row[ROW_PARAM]
for ax_i in axes:
    col_idx = list(pivot_pnl.columns).index(best_col_val) if best_col_val in pivot_pnl.columns else None
    row_idx = list(pivot_pnl.index).index(best_row_val) if best_row_val in pivot_pnl.index else None
    if col_idx is not None and row_idx is not None:
        ax_i.plot(col_idx, row_idx, 'k*', markersize=15, markeredgecolor='white', markeredgewidth=1)

plt.suptitle(f"Plateau Analysis — best combo marked with ★", fontsize=13)
plt.tight_layout()
plt.show()

---
## Part 2: Walk-Forward Analysis

The real test: optimize on one chunk of data, trade the *next* chunk,
slide forward, repeat. If the optimizer's picks keep making money on
unseen data, the signal is real. If they don't, you're curve-fitting.

Default: 50% train, 12.5% test → 4 non-overlapping folds.
Adjust `train_pct` and `test_pct` based on your data length.

In [ ]:
# ── Cell 6: Run walk-forward ───────────────────────────────────────
#
# This runs a full parameter sweep PER FOLD on the training window,
# then tests the winner on unseen data.  Takes a few minutes.

wf_results = run_walk_forward(
    venue=VENUE,
    instrument=instrument,
    bars=bars,
    starting_capital=STARTING_CAPITAL,
    param_combos=PARAM_COMBOS,
    strategy_factory=strategy_factory,
    train_pct=0.50,     # 50% of bars for training
    test_pct=0.125,     # 12.5% for testing → ~4 folds
    select_by="total_pnl",
)

In [ ]:
# ── Cell 7: Walk-forward results ──────────────────────────────────

if not wf_results.empty:
    # Summary table
    display_cols = [c for c in wf_results.columns if c not in (
        "train_start", "train_end", "test_start", "test_end",
        "train_bars", "test_bars", "oos_error",
    )]
    print("=== Walk-Forward Fold Results ===")
    display(wf_results[display_cols])

    # ── OOS PnL bar chart ──
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: OOS PnL per fold
    ax = axes[0]
    colors = ["#26a69a" if p > 0 else "#ef5350" for p in wf_results["oos_pnl"]]
    ax.bar(wf_results["fold"].astype(str), wf_results["oos_pnl"], color=colors)
    ax.axhline(0, color="white", linewidth=0.5, alpha=0.3)
    ax.set_xlabel("Fold")
    ax.set_ylabel(f"Out-of-Sample PnL ({CURRENCY})")
    ax.set_title("OOS PnL by Fold")
    ax.set_facecolor("#131722")
    fig.patch.set_facecolor("#131722")
    ax.tick_params(colors="#d1d4dc")
    ax.xaxis.label.set_color("#d1d4dc")
    ax.yaxis.label.set_color("#d1d4dc")
    ax.title.set_color("#d1d4dc")

    # Right: In-sample vs OOS PnL comparison
    ax = axes[1]
    x = np.arange(len(wf_results))
    w = 0.35
    ax.bar(x - w/2, wf_results["in_sample_pnl"], w, label="In-sample", color="#2196f3", alpha=0.7)
    ax.bar(x + w/2, wf_results["oos_pnl"], w, label="Out-of-sample", color="#f5c518", alpha=0.7)
    ax.axhline(0, color="white", linewidth=0.5, alpha=0.3)
    ax.set_xticks(x)
    ax.set_xticklabels(wf_results["fold"].astype(str))
    ax.set_xlabel("Fold")
    ax.set_ylabel(f"PnL ({CURRENCY})")
    ax.set_title("In-Sample vs Out-of-Sample")
    ax.legend()
    ax.set_facecolor("#131722")
    ax.tick_params(colors="#d1d4dc")
    ax.xaxis.label.set_color("#d1d4dc")
    ax.yaxis.label.set_color("#d1d4dc")
    ax.title.set_color("#d1d4dc")

    plt.tight_layout()
    plt.show()

    # ── Param drift ──
    param_cols = [c for c in wf_results.columns if c.startswith("best_")]
    if param_cols:
        print("\n=== Selected Params Per Fold ===")
        display(wf_results[["fold"] + param_cols + ["in_sample_pnl", "oos_pnl"]])

---
## Part 3: Bootstrap Confidence Intervals

Your strategy made $X on this data. But how *confident* should you be in
that number? If 80% of the PnL came from 2 lucky trades, the confidence
interval will be wide. If it came from 40 consistent small wins, it'll be tight.

We run a single backtest with the best params, extract per-trade PnL,
and resample 10,000 times to build a distribution of possible outcomes.

In [ ]:
# ── Cell 8: Run single backtest + extract per-trade PnL ───────────
#
# Uses the best params from the full-data sweep (Cell 3).
# If walk-forward suggests different params are more robust,
# override below: VALIDATE_PARAMS["fast"] = 20

VALIDATE_PARAMS = dict(best_params)
param_str = ", ".join(f"{k}={v}" for k, v in VALIDATE_PARAMS.items())
print(f"Validating: {param_str}")

eng = make_engine(VENUE, instrument, bars, STARTING_CAPITAL)
strategy_factory(eng, VALIDATE_PARAMS)
eng.run()

# Extract per-position PnL
positions_report = eng.trader.generate_positions_report()
positions = eng.cache.position_snapshots() + eng.cache.positions()
eng.dispose()

if positions_report.empty:
    print("No positions — nothing to bootstrap.")
    trade_pnls = np.array([])
else:
    # Parse PnL from positions report — stored as "123.45 USDC" strings
    def parse_pnl(val):
        if val is None or str(val) in ("None", "nan", "NaT"):
            return np.nan
        try:
            return float(str(val).split()[0])
        except (ValueError, IndexError):
            return np.nan

    trade_pnls = positions_report["realized_pnl"].apply(parse_pnl).dropna().values

    n_trades  = len(trade_pnls)
    total_pnl = trade_pnls.sum()
    winners   = (trade_pnls > 0).sum()
    losers    = (trade_pnls < 0).sum()

    print(f"Trades: {n_trades}  (W:{winners} / L:{losers})")
    print(f"Total PnL: {total_pnl:,.2f}")
    print(f"Mean per trade: {trade_pnls.mean():,.2f}")
    print(f"Median per trade: {np.median(trade_pnls):,.2f}")
    print(f"Std per trade: {trade_pnls.std():,.2f}")

    # Concentration check — what % of PnL comes from the best N trades?
    if total_pnl > 0:
        sorted_pnls = np.sort(trade_pnls)[::-1]
        cumsum = np.cumsum(sorted_pnls)
        top3_pct = cumsum[min(2, n_trades - 1)] / total_pnl * 100 if total_pnl > 0 else 0
        print(f"\nTop 3 trades account for {top3_pct:.0f}% of total PnL")
        if top3_pct > 75:
            print("🚩 Concentrated — result depends heavily on a few trades.")
        elif top3_pct > 50:
            print("⚠️  Moderately concentrated.")
        else:
            print("✅ Well distributed across trades.")

In [ ]:
# ── Cell 9: Bootstrap resampling ──────────────────────────────────

if len(trade_pnls) >= 5:
    np.random.seed(42)  # reproducible
    n = len(trade_pnls)
    # Vectorized bootstrap: resample n trades with replacement, N times
    indices = np.random.randint(0, n, size=(N_RESAMPLES, n))
    resampled_totals = trade_pnls[indices].sum(axis=1)

    # Percentiles
    p5  = np.percentile(resampled_totals, 5)
    p25 = np.percentile(resampled_totals, 25)
    p50 = np.percentile(resampled_totals, 50)
    p75 = np.percentile(resampled_totals, 75)
    p95 = np.percentile(resampled_totals, 95)
    actual = trade_pnls.sum()
    prob_positive = (resampled_totals > 0).mean() * 100

    print(f"Bootstrap: {N_RESAMPLES:,} resamples of {n} trades")
    print(f"")
    print(f"  5th percentile:   {p5:>10,.2f}  (worst realistic case)")
    print(f"  25th percentile:  {p25:>10,.2f}")
    print(f"  Median:           {p50:>10,.2f}")
    print(f"  75th percentile:  {p75:>10,.2f}")
    print(f"  95th percentile:  {p95:>10,.2f}  (best realistic case)")
    print(f"")
    print(f"  Actual PnL:       {actual:>10,.2f}")
    print(f"  P(profitable):    {prob_positive:>9.1f}%")
    print()
    print(f"  90% CI: [{p5:,.2f}, {p95:,.2f}]")

    if prob_positive >= 90:
        print("\n✅ High confidence — profitable in 90%+ of resamples.")
    elif prob_positive >= 70:
        print("\n⚠️  Moderate confidence — profitable but depends on trade mix.")
    else:
        print("\n🚩 Low confidence — result is fragile, could easily be negative.")

    # ── Histogram ──
    fig, ax = plt.subplots(figsize=(12, 5))
    fig.patch.set_facecolor("#131722")
    ax.set_facecolor("#131722")

    ax.hist(resampled_totals, bins=80, color="#2196f3", alpha=0.7, edgecolor="none")

    # Mark percentiles
    for pval, plabel, style in [
        (p5,  "5th",  {"color": "#ef5350", "linestyle": "--"}),
        (p50, "50th", {"color": "#d1d4dc", "linestyle": "-"}),
        (p95, "95th", {"color": "#26a69a", "linestyle": "--"}),
    ]:
        ax.axvline(pval, **style, linewidth=1.5, label=f"{plabel}: {pval:,.0f}")

    # Mark actual
    ax.axvline(actual, color="#f5c518", linewidth=2, label=f"Actual: {actual:,.0f}")

    # Zero line
    ax.axvline(0, color="white", linewidth=0.5, alpha=0.3)

    ax.set_xlabel(f"Total PnL ({CURRENCY})", color="#d1d4dc")
    ax.set_ylabel("Frequency", color="#d1d4dc")
    ax.set_title(
        f"Bootstrap Distribution — {n} trades resampled {N_RESAMPLES:,}×\n"
        f"90% CI: [{p5:,.0f}, {p95:,.0f}]   P(profit): {prob_positive:.0f}%",
        color="#d1d4dc", fontsize=12,
    )
    ax.tick_params(colors="#d1d4dc")
    ax.legend(facecolor="#1e222d", edgecolor="#2a2e39", labelcolor="#d1d4dc")
    plt.tight_layout()
    plt.show()

else:
    print(f"Only {len(trade_pnls)} trades — need at least 5 for meaningful bootstrap.")
    print("Run a longer backtest or use a more active strategy.")

In [ ]:
# ── Cell 10: Rolling Performance Windows ─────────────────────────────────
rolling_df = rolling_performance(positions, bars, window="90D")
display(rolling_df)

plot_rolling_pnl(rolling_df, currency=str(CURRENCY))

# Concentration check
if not rolling_df.empty:
    positive_windows = (rolling_df["pnl"] > 0).sum()
    total_windows = len(rolling_df)
    print(f"Positive windows: {positive_windows}/{total_windows} ({positive_windows/total_windows*100:.0f}%)")

In [ ]:
# ── Cell 11: Fee Sensitivity Sweep ──────────────────────────────────────
fee_df = run_fee_sweep(
    venue=VENUE,
    instrument=instrument,
    bars=bars,
    starting_capital=STARTING_CAPITAL,
    params=VALIDATE_PARAMS,
    strategy_factory=strategy_factory,
)
display(fee_df[["fee_bps", "total_pnl", "total_pnl_pct", "pnl_per_trade", "breakeven"]])

plot_fee_sensitivity(fee_df, currency=str(CURRENCY))

In [ ]:
# ── Cell 12: Regime Analysis ─────────────────────────────────────────────
regime_df = tag_regimes(bars, method="adx")
regime_perf = performance_by_regime(positions, regime_df)
display(regime_perf)

plot_regime_overlay(regime_df)

# Regime distribution
regime_counts = regime_df["regime"].value_counts(normalize=True)
print("Regime distribution (% of bars):")
for regime, pct in regime_counts.items():
    print(f"  {regime}: {pct*100:.1f}%")

---
## Summary: Go / No-Go

In [ ]:
# ── Cell 13: Go / No-Go assessment ────────────────────────────────
#
# Aggregates the three checks into a simple verdict.
# This is advisory — not a substitute for judgment.

param_str = ", ".join(f"{k}={v}" for k, v in VALIDATE_PARAMS.items())

print("=" * 60)
print(f"  VALIDATION SUMMARY")
print(f"  {INSTRUMENT_ID}  {BAR_INTERVAL}")
print(f"  {param_str}")
print("=" * 60)

checks = []

# 1. Plateau
if best_score >= 0.8:
    checks.append(("✅", "Plateau", f"Score {best_score:.2f} — robust region"))
elif best_score >= 0.5:
    checks.append(("⚠️", "Plateau", f"Score {best_score:.2f} — ridge, moderate risk"))
else:
    checks.append(("🚩", "Plateau", f"Score {best_score:.2f} — isolated spike, high overfit risk"))

# 2. Walk-forward
if not wf_results.empty:
    oos_profitable = (wf_results["oos_pnl"] > 0).sum()
    oos_total = len(wf_results)
    oos_total_pnl = wf_results["oos_pnl"].sum()
    if oos_profitable == oos_total and oos_total_pnl > 0:
        checks.append(("✅", "Walk-forward", f"{oos_profitable}/{oos_total} folds profitable, total OOS PnL {oos_total_pnl:,.2f}"))
    elif oos_profitable >= oos_total * 0.5 and oos_total_pnl > 0:
        checks.append(("⚠️", "Walk-forward", f"{oos_profitable}/{oos_total} folds profitable, total OOS PnL {oos_total_pnl:,.2f}"))
    else:
        checks.append(("🚩", "Walk-forward", f"{oos_profitable}/{oos_total} folds profitable, total OOS PnL {oos_total_pnl:,.2f}"))
else:
    checks.append(("⚠️", "Walk-forward", "No folds completed — need more data"))

# 3. Bootstrap
if len(trade_pnls) >= 5:
    if prob_positive >= 90:
        checks.append(("✅", "Bootstrap", f"P(profit)={prob_positive:.0f}%, 90% CI [{p5:,.0f}, {p95:,.0f}]"))
    elif prob_positive >= 70:
        checks.append(("⚠️", "Bootstrap", f"P(profit)={prob_positive:.0f}%, 90% CI [{p5:,.0f}, {p95:,.0f}]"))
    else:
        checks.append(("🚩", "Bootstrap", f"P(profit)={prob_positive:.0f}%, 90% CI [{p5:,.0f}, {p95:,.0f}]"))
else:
    checks.append(("⚠️", "Bootstrap", f"Only {len(trade_pnls)} trades — insufficient"))

# 4. Rolling windows
if not rolling_df.empty:
    pos_windows = (rolling_df["pnl"] > 0).sum()
    total_win = len(rolling_df)
    pct = pos_windows / total_win * 100
    if pct >= 60:
        checks.append(("✅", "Rolling", f"{pos_windows}/{total_win} windows profitable ({pct:.0f}%)"))
    elif pct >= 40:
        checks.append(("⚠️", "Rolling", f"{pos_windows}/{total_win} windows profitable ({pct:.0f}%)"))
    else:
        checks.append(("🚩", "Rolling", f"{pos_windows}/{total_win} windows profitable ({pct:.0f}%) — concentrated"))

# 5. Fee sensitivity
if not fee_df.empty:
    breakeven_rows = fee_df[fee_df["breakeven"]]
    if breakeven_rows.empty:
        checks.append(("🚩", "Fee sensitivity", "Not profitable at any fee level"))
    else:
        max_fee = breakeven_rows["fee_bps"].max()
        if max_fee >= 7.5:
            checks.append(("✅", "Fee sensitivity", f"Profitable up to {max_fee:.1f} bps — strong margin"))
        elif max_fee >= 4:
            checks.append(("⚠️", "Fee sensitivity", f"Profitable up to {max_fee:.1f} bps — moderate margin"))
        else:
            checks.append(("🚩", "Fee sensitivity", f"Breakeven at {max_fee:.1f} bps — thin margin"))

# 6. Regime
if not regime_perf.empty:
    ranging = regime_perf[regime_perf["regime"] == "RANGING"]
    trending = regime_perf[regime_perf["regime"] == "TRENDING"]
    if not ranging.empty and not trending.empty:
        ranging_pnl = float(ranging["pnl"].iloc[0])
        trending_pnl = float(trending["pnl"].iloc[0])
        if trending_pnl > 0 and trending_pnl > abs(ranging_pnl):
            checks.append(("✅", "Regime", f"Trending +{trending_pnl:,.0f} > Ranging {ranging_pnl:,.0f}"))
        elif trending_pnl > 0:
            checks.append(("⚠️", "Regime", f"Trending +{trending_pnl:,.0f}, Ranging {ranging_pnl:,.0f} — net depends on mix"))
        else:
            checks.append(("🚩", "Regime", f"Trending {trending_pnl:,.0f}, Ranging {ranging_pnl:,.0f} — no clear edge"))
            
print()
for icon, name, detail in checks:
    print(f"  {icon} {name:15s} {detail}")

n_pass = sum(1 for icon, _, _ in checks if icon == "✅")
n_warn = sum(1 for icon, _, _ in checks if icon == "⚠️")
n_fail = sum(1 for icon, _, _ in checks if icon == "🚩")

print()
if n_fail > 0:
    print("  VERDICT: 🚩 DO NOT paper trade yet. Address the red flags first.")
elif n_warn > 0:
    print("  VERDICT: ⚠️  PROCEED WITH CAUTION. Monitor closely in paper trading.")
else:
    print("  VERDICT: ✅ READY for paper trading.")
print()
print("  Remember: expect 30-40% haircut from backtest to live.")
print("=" * 60)

In [ ]:
# ── Save notebook snapshot ────────────────────────────────────────
# Uncomment after running all cells (Ctrl+S first).
#save_notebook("validate_strategy.ipynb", RESULT_NAME, category="validate")
#save_notebook_html("validate_strategy.ipynb", RESULT_NAME, category="validate")